# Interference: when amplitudes cancel
**Quantum Computing · Week 3 (The maths of qubits, and interference) · Day 3**

Lesson page: https://sara-aminpour.com/tutorials/qc/w3d3.html

In [ ]:
# On Google Colab, this installs qiskit and qiskit-aer (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 qiskit-aer==0.17.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2 qiskit-aer==0.17.2

To get the probability of an outcome, **add the amplitudes of every path** that leads to it,
*then* square. Because amplitudes can be negative (or complex), paths can cancel.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

recipes = {"H": ["h"], "H, H": ["h", "h"], "H, Z, H": ["h", "z", "h"]}
for name, gates in recipes.items():
    qc = QuantumCircuit(1)
    for g in gates:
        getattr(qc, g)(0)
    sv = Statevector(qc)
    print(f"{name:>8}: amplitudes {np.round(sv.data.real, 3)}  ->  P(0) = {sv.probabilities()[0]:.2f}")

## The path sum, by hand
After one H, the qubit is half on the "0 path" and half on the "1 path" (amplitude $1/\sqrt2$ each).
The second H sends each path to both outcomes. For outcome 1 the two contributions are $+\tfrac12$ and $-\tfrac12$.

In [ ]:
H = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
start = np.array([1, 0])
middle = H @ start
end = H @ middle

print("after one H :", middle.round(3))
print("after two H :", end.round(3))
print("to outcome 1: from the 0 path", round(H[1, 0] * middle[0], 3), " + from the 1 path", round(H[1, 1] * middle[1], 3))

## Turn the phase knob
Put a phase gate `P(φ)` between the two H gates. It multiplies the 1 path by $e^{i\varphi}$.
The chance of measuring 0 swings between 1 and 0: interference **fringes**.

In [ ]:
import matplotlib.pyplot as plt
from qiskit.circuit import Parameter

phi = Parameter("φ")
qc = QuantumCircuit(1)
qc.h(0)
qc.p(phi, 0)
qc.h(0)

phis = np.linspace(0, 4 * np.pi, 121)
p0 = [Statevector(qc.assign_parameters({phi: v})).probabilities()[0] for v in phis]
plt.plot(phis, p0, "o", ms=3.5, label="Qiskit")
plt.plot(phis, np.cos(phis / 2) ** 2, label="cos²(φ/2)")
plt.xlabel("phase φ (radians)")
plt.ylabel("P(0)")
plt.title("Interference fringes: H, then P(φ), then H")
plt.legend(loc="lower right")
plt.show()

In [ ]:
qc.draw("mpl", style="clifford")

## Measured with shots

In [ ]:
from qiskit.primitives import StatevectorSampler

circuits = []
for value in [0, np.pi / 2, np.pi]:
    c = qc.assign_parameters({phi: value})
    c.measure_all()
    circuits.append(c)

results = StatevectorSampler(seed=1).run(circuits, shots=1000).result()
for label, res in zip(["0", "π/2", "π"], results):
    print(f"φ = {label:>3}:  {res.data.meas.get_counts()}")

## Your turn: spy on the path
What if we peek at the qubit **between** the two H gates? Add a measurement in the middle and compare $P(0)$ at the
end, with and without the spy. (The basic sampler can't measure in the middle of a circuit, so we use Aer.)

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator

sim = AerSimulator(seed_simulator=7)

def p0_final(phi_value, spy):
    c = QuantumCircuit(1, 2)            # two classical bits: one for the spy, one for the end
    c.h(0)
    if spy:
        pass                            # TODO: measure qubit 0 into classical bit 0
    c.p(phi_value, 0)
    c.h(0)
    c.measure(0, 1)                     # the final measurement goes into classical bit 1
    counts = sim.run(transpile(c, sim), shots=2000).result().get_counts()
    # bitstrings read "bit1 bit0", so the final result is the left character
    return sum(n for b, n in counts.items() if b[0] == "0") / 2000

for value in [0, np.pi / 2, np.pi]:
    print(f"phi = {value:.2f}:  P(0) without the spy {p0_final(value, False):.2f}   with the spy {p0_final(value, True):.2f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
def p0_final(phi_value, spy):
    c = QuantumCircuit(1, 2)
    c.h(0)
    if spy:
        c.measure(0, 0)                 # peek: which path is the qubit on?
    c.p(phi_value, 0)
    c.h(0)
    c.measure(0, 1)
    counts = sim.run(transpile(c, sim), shots=2000).result().get_counts()
    return sum(n for b, n in counts.items() if b[0] == "0") / 2000

for value in [0, np.pi / 2, np.pi]:
    print(f"phi = {value:.2f}:  P(0) without the spy {p0_final(value, False):.2f}   with the spy {p0_final(value, True):.2f}")
print("The spy collapses the qubit to one path, so there is nothing left to interfere: always about 50/50.")